# 🤖 Amazon Bedrock AgentCore Browser Tool

Welcome to this hands-on lab where you'll learn to integrate Amazon Bedrock AgentCore's browser tool into your agents. This tool enables agents to interact with web pages through automated browser sessions powered by Amazon NovaAct.

![](../images/browser_tool_arch.png)

## Learning Objectives

- Integrate Amazon Bedrock AgentCore's browser tool with agents
- Understand asynchronous tool execution patterns
- Build agents that can search and interact with websites like Amazon.com
- Handle browser automation results in conversational flows

## Why a Browser Tool?

Some information lives on websites with no API — product prices, dynamic content, pages behind authentication. Tavily can search the web, but it can't navigate, click buttons, or interact with page elements. The browser tool gives agents full page interaction via headless browser sessions powered by Amazon NovaAct (a framework for driving browser behavior with LLMs).

### Prerequisites

This notebook requires a NovaAct API key. Request one at [nova.amazon.com/act](https://nova.amazon.com/act). Add it to a `.env` file in this directory:

```
NOVA_ACT_API_KEY=your-key-here
```

In [ ]:
# Import dependencies. 

import boto3
import json
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('.env')

# Verify NOVA_ACT_API_KEY is loaded
if 'NOVA_ACT_API_KEY' not in os.environ:
    raise ValueError("NOVA_ACT_API_KEY not found in environment. Please add it to the .env file")

# Initialize the Bedrock client
session = boto3.Session()
bedrock = session.client(service_name='bedrock-runtime')

print("✅ Setup complete!")
print(f"✅ NOVA_ACT_API_KEY loaded from .env")

In the [Module 3 Lab 3 (Agent Tools)](../../module3/notebooks/3_agent_tools.ipynb), we created an Agent implementation with features like memory and tool calling. We'll now extend this concept by integrating Amazon Bedrock AgentCore's browser tool, which enables agents to interact with web pages through automated browser sessions.

## Reuse what we wrote in the previous lab. 

In [ ]:
from labs_common import LabsBasePrompt as BasePrompt
from agentic_platform.core.models.memory_models import Message, SessionContext, ToolResult, ToolCall
from agentic_platform.core.models.llm_models import LLMResponse, LLMRequest
from agentic_platform.core.converter.llm_request_converters import ConverseRequestConverter
from agentic_platform.core.converter.llm_response_converters import ConverseResponseConverter
from agentic_platform.core.models.api_models import AgenticRequest, AgenticResponse
from typing import Dict, Any, Optional, List, Type, Callable

# Helper to construct request
def construct_request(user_message: str, conversation_id: str=None) -> AgenticRequest:
    return AgenticRequest.from_text(
        text=user_message, 
        **{'session_id': conversation_id}
    )

# Helper function to call Bedrock. Passing around JSON is messy and error prone.
def call_bedrock(request: LLMRequest) -> LLMResponse:
    kwargs: Dict[str, Any] = ConverseRequestConverter.convert_llm_request(request)
    # Call Bedrock
    converse_response: Dict[str, Any] = bedrock.converse(**kwargs)
    # Get the model's text response
    return ConverseResponseConverter.to_llm_response(converse_response)

class MemoryClient:
    """Manages conversations"""
    def __init__(self):
        self.conversations: Dict[str, SessionContext] = {}

    def upsert_conversation(self, conversation: SessionContext, conversation_id: str=None) -> bool:
        if conversation_id:
            self.conversations[conversation_id] = conversation
        else:
            self.conversations[conversation.session_id] = conversation

    def get_or_create_conversation(self, conversation_id: str=None) -> SessionContext:
        return self.conversations.get(conversation_id, SessionContext()) if conversation_id else SessionContext()
    
memory_client: MemoryClient = MemoryClient()

### Agent Infrastructure (from Module 3)

The `ToolCallingAgent` below is reused from Module 3 Lab 3. It provides:
- **Conversation memory** — maintains message history across turns
- **Bedrock integration** — calls the Converse API with tool definitions
- **Tool execution loop** — calls tools until the model produces a final text response

You don't need to read through this cell — it's infrastructure. Focus on the browser-specific code that follows.

In [ ]:
from pydantic import BaseModel

# Import our tool and LLM models
from agentic_platform.core.models.memory_models import TextContent
from agentic_platform.core.models.tool_models import ToolSpec
from agentic_platform.core.models.llm_models import LLMRequest, LLMResponse


class ToolCallingAgent:
    # This is new, we're adding tools in the constructor to bind them to the agent.
    # Don't get too attached to this idea, it'll change as we get into MCP.
    def __init__(self, tools: List[ToolSpec], prompt: BasePrompt):
        self.tools: List[ToolSpec] = tools
        self.conversation: SessionContext = SessionContext()
        self.prompt: BasePrompt = prompt

    def call_llm(self) -> LLMResponse:
        # Create LLM request
        request: LLMRequest = LLMRequest(
            system_prompt=self.prompt.system_prompt,
            messages=self.conversation.get_messages(),
            model_id=self.prompt.model_id,
            hyperparams=self.prompt.hyperparams,
            tools=self.tools
        )

        # Call the LLM.
        response: LLMResponse = call_bedrock(request)
        # Append the llms response to the conversation.
        self.conversation.add_message(Message(
            role="assistant",
            text=response.text,
            tool_calls=response.tool_calls
        ))
        # Return the response.
        return response
    
    def execute_tools(self, llm_response: LLMResponse, session_id:str) -> List[ToolResult]:
        """Call tools and return the results."""
        # It's possible that the model will call multiple tools.
        tool_results: List[ToolResult] = []
        # Iterate over the tool calls and call the tool.
        for tool_invocation in llm_response.tool_calls:
            # Get the tool spec for the tool call.
            tool: ToolSpec = next((t for t in self.tools if t.name == tool_invocation.name), None)
            # Call the tool.
            input_data: BaseModel = tool.model.model_validate(tool_invocation.arguments)
            # Dynamically inject session_id if the model has it
            if hasattr(input_data, "session_id"):
                setattr(input_data, "session_id", session_id)
            function_result: str = str(tool.function(input_data))
            tool_response: ToolResult = ToolResult(
                id=tool_invocation.id,
                content=[TextContent(text=function_result)],
                isError=False
            )

            print(f"Tool response: {tool_response}")

            # Add the tool result to the list.
            tool_results.append(tool_response)

        # Add the tool results to the conversation
        message: Message = Message(role="user", tool_results=tool_results)
        self.conversation.add_message(message)
        
        # Return the tool results even though we don't use it.
        return tool_results
    
    def invoke(self, request: AgenticRequest) -> AgenticResponse:
        # Get or create conversation
        self.conversation = memory_client.get_or_create_conversation(request.session_id)
        # Add user message to conversation
        self.conversation.add_message(request.message)

        # Keep calling LLM until we get a final response
        while True:
            # Call the LLM
            response: LLMResponse = self.call_llm()
            
            # If the model wants to use tools
            if response.stop_reason == "tool_use":
                # Execute the tools
                self.execute_tools(response,request.session_id)
                # Continue the loop to get final response
                continue
            
            # If we get here, it's a final response 
            break

        # Save updated conversation
        memory_client.upsert_conversation(self.conversation,request.session_id)

        # Return our own type.
        return AgenticResponse(
            message=self.conversation.messages[-1], # Just return the last message
            session_id=request.session_id if request.session_id else self.conversation.session_id
        )

## Example: Amazon.com Search Agent


In this example, we demonstrate how to integrate a tool that performs **product searches on Amazon.com** using the `browser_client` tool from the [Amamazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-tool.html) module.

This tool uses a **headless browser session** to navigate Amazon.com, perform a product search, and return the result asynchronously. The actual browser automation is powered by **Amazon NovaAct**, a framework for driving browser behavior with LLMs. Please request an API key [here](https://nova.amazon.com/act). 

Once the background browser task completes, the result is saved into memory and can be retrieved during a follow-up turn in the conversation.

This tool is ideal for use cases such as:
- Looking up the price of a product on Amazon
- Comparing different models or versions


In [ ]:
class AgentPrompt(BasePrompt):
    system_prompt: str = (
        "You are a helpful assistant with access to a browser search tool.\n\n"
        "TOOL USAGE RULES:\n"
        "1. Use 'browser_search' to search Amazon.com for product information. The tool takes 30-60 seconds but returns the actual result directly.\n"
        "2. Only report data returned by tools. Never fabricate prices, facts, or URLs.\n"
        "3. Present the tool result clearly to the user."
    )
    user_prompt: str = "{user_message}"

### Cross-Platform Design: Subprocess Worker Pattern

Playwright (used by Nova Act) needs to spawn a child process for its browser driver. Inside Jupyter on Windows, the asyncio event loop doesn't support `create_subprocess_exec`, which causes a `NotImplementedError`.

To work around this on **both Windows and Linux**, we use a **subprocess worker pattern**:

1. The notebook spawns `../utils/browser_worker.py` as a separate Python process via `subprocess.run()`
2. The child process gets its own clean event loop where Playwright works normally
3. Results are passed back via a temporary JSON file

This keeps the notebook code simple while being fully cross-platform.

### Synchronous Subprocess Pattern

The browser tool **blocks** while `browser_worker.py` runs as a subprocess (~30–60 seconds). This is simpler than an async pattern and works reliably because:

1. The model calls `browser_search` → the tool function runs `subprocess.run()` (blocking)
2. The subprocess drives the browser via NovaAct, writes result to a temp JSON file
3. The tool reads the file and returns the result as a string
4. The model receives the price directly in its tool result — no polling, no second tool

**Why a subprocess?** Playwright (used by NovaAct) needs `create_subprocess_exec`, which fails inside Jupyter's nested asyncio loop on Windows. A separate process gets its own clean event loop.

In [ ]:
import os
import sys
import json
import subprocess
import tempfile


def browser_search(request: str, session_id: str) -> str:
    """
    Runs a browser search synchronously via subprocess.
    Blocks for 30-60 seconds while NovaAct drives the browser, then returns the result.
    """
    print(f"🔍 Starting browser search: {request}")
    output_file = os.path.join(tempfile.gettempdir(), f"browser_result_{session_id}.json")

    # browser_worker.py lives in ../utils/ relative to this notebook
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    worker_script = os.path.join(notebook_dir, "..", "utils", "browser_worker.py")
    worker_script = os.path.normpath(worker_script)
    if not os.path.exists(worker_script):
        worker_script = os.path.normpath(os.path.join(os.getcwd(), "..", "utils", "browser_worker.py"))

    try:
        proc = subprocess.run(
            [sys.executable, worker_script, "--request", request, "--output", output_file],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=120,
        )

        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)

        if os.path.exists(output_file):
            with open(output_file, "r", encoding="utf-8") as f:
                result_data = json.load(f)
            os.remove(output_file)

            if result_data.get("success"):
                print(f"✅ Browser result: {result_data['result']}")
                return f"Search completed on Amazon.com. Result: {result_data['result']}"
            else:
                return f"Browser search failed: {result_data.get('error', 'unknown error')}"
        else:
            return f"Browser worker did not produce output. Exit code: {proc.returncode}"

    except subprocess.TimeoutExpired:
        return "Browser search timed out after 120 seconds."
    except Exception as e:
        return f"Browser search error: {e}"

In [ ]:
from pydantic import BaseModel

class BrowserSearchInput(BaseModel):
    request: str
    session_id: str

In [ ]:
from agentic_platform.core.models.tool_models import ToolSpec

browser_tool_spec = ToolSpec(
    name="browser_search",
    description="Searches Amazon.com using a headless browser powered by NovaAct. Takes 30-60 seconds. Returns the search result directly (e.g., a product price).",
    model=BrowserSearchInput,
    function=lambda args: browser_search(args.request, args.session_id)
)

In [ ]:
memory_client = MemoryClient()
agent = ToolCallingAgent(
    tools=[browser_tool_spec],
    prompt=AgentPrompt()
)

When you invoke the agent, it will launch a background task to search for products. To see this in action:

1. Navigate to the AWS Console and go to the Amazon Bedrock section
![](../images/go_to_bedrock_agentcore_console.png)


2. In the Browser Tool tab, you'll see your agent's browser sessions
   ![](../images/browser_use_tab.png)

3. You can see the built-in browser sandbox that AgentCore provides
   ![](../images/aws_built_in_browser_sandbox.png)

4. Click on "View Live Session" to see the browser in action
   ![](../images/click_view_live_session.png)

5. Watch as the agent interacts with the browser to search for products
   ![](../images/watch_the_agent_interact_with_browser.png)

### Running the Agent

The cell below will take 30–60 seconds to complete — the tool blocks while NovaAct drives the browser. You'll see live progress output from the subprocess, then the final response from the model with the actual price.

In [ ]:
session_id = "shopping-session-1"

req = construct_request("Find the price of an Apple Watch Series 10 on Amazon", session_id)

response = agent.invoke(req)
print(response.message.model_dump_json(indent=2))

### Follow-up Question (Conversation Memory)

Since the agent stores conversation history, a follow-up question can reference the previous result without re-running the browser search.

In [ ]:
req2 = construct_request("Can you summarize what you found about the Apple Watch price?", session_id)

response = agent.invoke(req2)
print(response.message.model_dump_json(indent=2))

## Summary

In this lab, you learned how to integrate Amazon Bedrock AgentCore's browser tool into your agents. Key takeaways:

- **Browser Automation**: Agents can interact with web pages through headless browser sessions powered by Amazon NovaAct
- **Asynchronous Execution**: Browser tools run in the background and save results to memory for later retrieval
- **Real-world Integration**: Demonstrated product search capabilities on Amazon.com
- **Conversational Flow**: Handled browser automation results within multi-turn conversations

This pattern enables agents to access real-time web content and perform complex web interactions beyond simple API calls.

## Additional Resources

- [Amazon Bedrock AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/)
- [Browser Tool Reference](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-tool.html)
- [Amazon NovaAct Framework](https://nova.amazon.com/act)
- [Module 3 Lab 3: Agent Tools](../../module3/notebooks/3_agent_tools.ipynb)

This concludes Module 4. In Module 5, we shift focus to production infrastructure — LLM gateways, streaming and evaluation at scale.